# Dual-Pol PolSAR Decomposition Tutorial

This notebook demonstrates polarimetric decompositions on **dual-polarization** SAR data.
Dual-pol data (co-pol + cross-pol, e.g. HH+HV or VV+VH) is the most common mode for
operational SAR missions such as NISAR frequency B, Sentinel-1, and RADARSAT-2.

**Algorithms demonstrated:**
1. `DualPolHAlpha` — Entropy / Alpha / Anisotropy decomposition (Cloude 2007)
2. Dual-pol RVI (`DpRVI`) — Radar Vegetation Index for dual-pol (placeholder)

**Data:** NISAR RSLC (HH+HV channels from frequency A, or synthetic fallback).

> **Full-pol H/Alpha:** For quad-pol data (HH/HV/VH/VV) see
> `full_pol_decomposition_tutorial.ipynb`, which covers the 3×3 coherency matrix
> and the full Cloude-Pottier three-parameter decomposition.

---

## Theory — Dual-Pol H/Alpha Decomposition

### 2×2 Coherency Matrix

For a dual-pol system with co-pol $S_{co}$ and cross-pol $S_{cx}$, the Pauli
target vector is:

$$k_2 = \frac{1}{\sqrt{2}} \begin{bmatrix} S_{co} + S_{cx} \\ S_{co} - S_{cx} \end{bmatrix}$$

The 2×2 coherency matrix is:

$$[T_2] = \langle k_2 k_2^H \rangle = \frac{1}{N} \sum_{n} k_2^{(n)} (k_2^{(n)})^H$$

### Eigendecomposition

$$[T_2] = U_2 \Lambda U_2^H, \quad \Lambda = \mathrm{diag}(\lambda_1, \lambda_2), \quad \lambda_1 \geq \lambda_2 \geq 0$$

### Entropy

Scattering randomness:

$$H = -\sum_{i=1}^{2} p_i \log_2 p_i, \quad p_i = \frac{\lambda_i}{\lambda_1 + \lambda_2}$$

- $H \approx 0$: single dominant scattering mechanism (e.g. specular reflection)
- $H \approx 1$: fully random depolarising scatterer

### Alpha Angle

Dominant scattering mechanism:

$$\alpha = \sum_{i=1}^{2} p_i \alpha_i, \quad \alpha_i = \cos^{-1}|e_{i1}|$$

- $\alpha \approx 0°$: surface / Bragg scattering (bare soil, calm water)
- $\alpha \approx 45°$: volume scattering (vegetation)
- $\alpha \approx 90°$: double-bounce (urban, forest stems)

### Anisotropy

Secondary scattering contribution (requires at least 3 eigenvalues for full interpretation;
for dual-pol it is directly $A = (\lambda_1 - \lambda_2)/(\lambda_1 + \lambda_2)$):

$$A = \frac{\lambda_1 - \lambda_2}{\lambda_1 + \lambda_2}$$

### RGB Composite

| Channel | Component | Interpretation |
|---------|-----------|----------------|
| R (red) | $\alpha / 90°$ | scattering mechanism |
| G (green) | Span (log-stretched) | total power |
| B (blue) | $H$ | scattering randomness |

**References:**
- Cloude, S.R. (2007). *The dual polarisation entropy/alpha decomposition: a PALSAR case study.*
  Proc. 3rd Int. Workshop on Science and Applications of SAR Polarimetry and Pol-InSAR.
- Cloude, S.R. and Pottier, E. (1997). *An entropy based classification scheme for land
  applications of polarimetric SAR.* IEEE Trans. Geoscience and Remote Sensing, 35(1), 68–78.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
import gc

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

try:
    import grdl
except ImportError as exc:
    raise ImportError(
        "\n"
        "═══════════════════════════════════════════════════════════════\n"
        "  grdl is NOT installed in this Python environment.\n"
        "  Install with:  pip install -e '.[all]'\n"
        "  from the grdl repository root.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from exc

print(f"grdl {grdl.__version__} — ready")


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from grdl.IO.sar.nisar import NISARReader
from grdl.image_processing.decomposition import DualPolHAlpha

try:
    from grdk.viewers.dual_viewer import DualGeoViewer
    HAS_GRDK = True
    try:
        get_ipython().run_line_magic('gui', 'qt6')
    except (NameError, AttributeError):
        pass
except ImportError as exc:
    DualGeoViewer = None
    HAS_GRDK = False
    print(f"Warning: grdk import failed ({exc}). Falling back to matplotlib.")

def show_dual_rgb(left, right, title, left_label='Left', right_label='Right'):
    if HAS_GRDK:
        viewer = DualGeoViewer()
        viewer.set_mode('dual')
        viewer.setWindowTitle(title)
        viewer.set_array(left, pane=0)
        viewer.set_array(right, pane=1)
        viewer.show()
        print(f"GRDK viewer launched: {title}")
        return viewer

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=100)
    for ax, arr, label in zip(axes, (left, right), (left_label, right_label)):
        img = np.asarray(arr)
        if img.ndim == 3 and img.shape[0] in (1, 3, 4):
            img = np.moveaxis(img, 0, -1)
        if img.ndim == 3 and img.shape[-1] == 1:
            img = img[..., 0]
        ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
        ax.set_title(label)
        ax.axis('off')
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()
    print(f"Matplotlib fallback shown: {title}")
    return fig

print('Imports OK')


In [ ]:
# =============================================================================
# Configuration
# =============================================================================
nisar_file = Path(
    '/data/sar/slc/nisar/l1_rslc/'
    '20260105T055924_20260105T055931/'
    'NISAR_L1_PR_RSLC_009_116_D_054_2005_QPDH_A_20260105T055924_20260105T055931'
    '_X05010_N_P_J_001.h5'
)
frequency    = 'A'
co_pol       = 'HH'    # co-polarization channel
cross_pol    = 'HV'    # cross-polarization channel
chip_size    = 1024

window_size  = 7       # T2 spatial averaging window (odd)

In [ ]:
# =============================================================================
# Load dual-pol data or generate synthetic fallback
# =============================================================================
def _synthetic_dual_pol(rows=512, cols=512, seed=42):
    """Synthetic dual-pol SLC with distinct scene classes."""
    rng = np.random.default_rng(seed)
    # Background: moderate volume
    s_co  = (rng.standard_normal((rows, cols)) + 1j * rng.standard_normal((rows, cols))).astype(np.complex64)
    s_cx  = (0.4 * (rng.standard_normal((rows, cols)) + 1j * rng.standard_normal((rows, cols)))).astype(np.complex64)
    # Urban block (top-left quadrant): high co-pol, low cross-pol
    r0, r1, c0, c1 = 0, rows // 4, 0, cols // 4
    s_co[r0:r1, c0:c1]  = (2.0 * rng.standard_normal((r1-r0, c1-c0)) + 1j * 2.0 * rng.standard_normal((r1-r0, c1-c0))).astype(np.complex64)
    s_cx[r0:r1, c0:c1] *= 0.1
    # Vegetation (bottom-right quadrant): high cross-pol
    r0, r1, c0, c1 = rows*3//4, rows, cols*3//4, cols
    s_cx[r0:r1, c0:c1] = (1.5 * rng.standard_normal((r1-r0, c1-c0)) + 1j * 1.5 * rng.standard_normal((r1-r0, c1-c0))).astype(np.complex64)
    return s_co, s_cx

if nisar_file.exists():
    reader = NISARReader(nisar_file, frequency=frequency, polarizations='all')
    meta = reader.metadata
    N = chip_size
    r0 = max(0, (meta.rows - N) // 2)
    c0 = max(0, (meta.cols - N) // 2)
    cube = reader.read_chip(r0, r0 + N, c0, c0 + N)
    reader.close()
    pol_index = {cm.polarization: i for i, cm in enumerate(meta.channel_metadata)}
    s_co  = cube[pol_index[co_pol]]
    s_cx  = cube[pol_index[cross_pol]]
    print(f'NISAR {co_pol}+{cross_pol}: {s_co.shape}')
    del cube
    gc.collect()
else:
    N = chip_size
    s_co, s_cx = _synthetic_dual_pol(rows=N, cols=N)
    print(f'Synthetic dual-pol: {s_co.shape}')

---
## 1. DualPolHAlpha — Entropy / Alpha / Anisotropy

In [ ]:
# =============================================================================
# DualPolHAlpha decomposition
# =============================================================================
dp = DualPolHAlpha(window_size=window_size)
components = dp.decompose_dual(s_co, s_cx)

H     = components['entropy']     # [0, 1]
alpha = components['alpha']       # degrees [0, 90]
A     = components['anisotropy']  # [0, 1]
span  = components['span']        # total power

print(f'H     range: [{H.min():.3f}, {H.max():.3f}]')
print(f'alpha range: [{alpha.min():.1f}°, {alpha.max():.1f}°]')
print(f'A     range: [{A.min():.3f}, {A.max():.3f}]')

In [ ]:
# =============================================================================
# Diagnostic matplotlib overview (individual component maps)
# =============================================================================
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].imshow(H, vmin=0, vmax=1, cmap='hot')
axes[0].set_title('Entropy H')
plt.colorbar(axes[0].images[0], ax=axes[0])

axes[1].imshow(alpha, vmin=0, vmax=90, cmap='RdYlBu_r')
axes[1].set_title('Alpha α (°)')
plt.colorbar(axes[1].images[0], ax=axes[1])

axes[2].imshow(A, vmin=0, vmax=1, cmap='viridis')
axes[2].set_title('Anisotropy A')
plt.colorbar(axes[2].images[0], ax=axes[2])

span_db = 10 * np.log10(np.maximum(span, 1e-10))
axes[3].imshow(span_db, cmap='gray',
               vmin=np.percentile(span_db, 2), vmax=np.percentile(span_db, 98))
axes[3].set_title('Span (dB)')

for ax in axes:
    ax.axis('off')
fig.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# RGB composite: R=alpha/90, G=span(dB stretch), B=H
# =============================================================================
def stretch(x, lo_pct=2, hi_pct=98):
    lo, hi = np.percentile(x, [lo_pct, hi_pct])
    return np.clip((x - lo) / max(hi - lo, 1e-8), 0.0, 1.0).astype(np.float32)

rgb, rgb_meta = dp.to_rgb(components)
print(f'to_rgb output: {rgb.shape}  ({" / ".join(cm.name for cm in rgb_meta.channel_metadata)})')

# Pauli-colour reference
r_p = stretch(np.abs(s_co - s_cx))
g_p = stretch(np.abs(s_cx))
b_p = stretch(np.abs(s_co + s_cx))
rgb_pauli = np.stack([r_p, g_p, b_p], axis=0)

viewer_ha = show_dual_rgb(
    rgb_pauli,
    rgb,
    f'DualPolHAlpha (win={window_size}) — Pauli RGB (L) vs H/α/span RGB (R)',
    left_label='Pauli-style RGB',
    right_label='DualPolHAlpha RGB',
)
print('Viewer: DualPolHAlpha')

---
## H-Alpha Plane

The H–α plane is a 2D scatter diagram that classifies pixels by scattering mechanism.
Nine zones (Cloude & Pottier 1997) partition the feasible region:

| Zone | H range | α range | Interpretation |
|------|---------|---------|----------------|
| Z1   | 0–0.5   | 0–42.5° | Surface scatter (low entropy) |
| Z4   | 0.5–0.9 | 0–40°   | Multiple-bounce (medium entropy) |
| Z7   | 0.9–1.0 | 0–55°   | Random isotropic scatter |
| Z2/5 | …       | …       | Vegetation / dipole |
| Z3/6 | …       | …       | Double-bounce |

For dual-pol, only a 2-zone layout (surface vs volume) is reliably defined.

In [ ]:
# =============================================================================
# H-Alpha plane (Cloude-Pottier classification diagram)
# =============================================================================
# Reference: Cloude & Pottier (1997), IEEE TGRS 35(1), Fig. 4
#
# ⚠ BASIS NOTE — dual-pol vs full-pol alpha are NOT directly comparable:
#   DualPolHAlpha builds the LEXICOGRAPHIC covariance matrix [C2] in the
#   raw (HH, HV) channel basis.  Alpha is computed from the eigenvectors of
#   [C2], so alpha≈0° means HH dominates (co-pol), alpha≈90° means HV
#   dominates (cross-pol).  This is NOT the Pauli-basis interpretation.
#
#   FullPolHAalpha (full-pol notebook) builds [T3] from the Pauli target
#   vector k=(HH+VV, HH−VV, 2HV)/√2 so alpha=0° means (HH+VV) surface
#   scattering and alpha=90° means double-bounce or volume.  Expect
#   alpha≈40–85° for L-band data over vegetation and urban areas.
#   Both observations are physically correct for their respective bases.
# Note: for dual-pol only 2 eigenvalues exist; zones Z1-Z9 are approximate.
H_flat     = H.ravel()
alpha_flat = alpha.ravel()

mask_ha = np.isfinite(H_flat) & np.isfinite(alpha_flat)
H_valid     = H_flat[mask_ha]
alpha_valid = alpha_flat[mask_ha]

fig, ax = plt.subplots(figsize=(8, 6))

hb = ax.hexbin(H_valid, alpha_valid, gridsize=120, cmap='inferno',
               mincnt=1, extent=[0, 1, 0, 90])
fig.colorbar(hb, ax=ax, label='Pixel count')

# Zone boundaries (Cloude-Pottier 9-zone scheme)
for h_line in [0.5, 0.9]:
    ax.axvline(h_line, color='gray', linewidth=0.8, linestyle='--', alpha=0.7)
for a_line in [42.5, 47.5]:
    ax.axhline(a_line, color='gray', linewidth=0.8, linestyle='--', alpha=0.7)

zone_labels = {
    (0.25, 21):  'Z9\nSurface',
    (0.25, 45):  'Z8\nDipole',
    (0.25, 69):  'Z7\nDihedral',
    (0.70, 21):  'Z6',
    (0.70, 45):  'Z5\nVegetation',
    (0.70, 69):  'Z4',
    (0.95, 21):  'Z3',
    (0.95, 45):  'Z2',
    (0.95, 69):  'Z1',
}
for (hpos, apos), label in zone_labels.items():
    ax.text(hpos, apos, label, color='cyan', fontsize=7,
            ha='center', va='center', alpha=0.8)

ax.set_xlabel('Entropy (H)')
ax.set_ylabel('Alpha (°)  [C2 lexicographic basis]')
ax.set_title('H-α Plane — Cloude-Pottier (dual-pol, [C2] basis)')
ax.set_xlim(0, 1)
ax.set_ylim(0, 90)
plt.tight_layout()
plt.show()

print(f'Valid pixels: {mask_ha.sum():,} / {mask_ha.size:,}')


---
## 2. Dual-pol RVI — Radar Vegetation Index (Placeholder)

The Dual-pol Radar Vegetation Index (DpRVI) uses the eigenvalue structure of $[T_2]$ to
quantify vegetation cover:

$$\mathrm{DpRVI} = 1 - \frac{\lambda_1}{\lambda_1 + \lambda_2}$$

- $\mathrm{DpRVI} \approx 0$: bare soil / low vegetation
- $\mathrm{DpRVI} \approx 1$: dense vegetation

**Not yet implemented** (Phase 5 of the PolSAR expansion plan).

**Reference:** Mandal et al. (2020). *Dual polarimetric radar vegetation index for crop
growth monitoring using Sentinel-1 SAR data.* Remote Sensing of Environment, 247, 111954.

In [ ]:
# =============================================================================
# DpRVI — placeholder (uncomment when Phase 5 is implemented)
# =============================================================================

# from grdl.image_processing.decomposition import DualPolRVI
#
# rvi_dp = DualPolRVI(window_size=window_size)
# rvi_components = rvi_dp.decompose_dual(s_co, s_cx)
# dprvi = rvi_components['dprvi']   # [0, 1]
#
# plt.figure(figsize=(6, 5))
# plt.imshow(dprvi, vmin=0, vmax=1, cmap='YlGn')
# plt.colorbar(label='DpRVI')
# plt.title('Dual-pol Radar Vegetation Index')
# plt.axis('off')
# plt.show()

print('DpRVI not yet implemented — placeholder only')